## 📦 Installation Requirements
Before running this notebook, ensure you have:

### 1. Install Ollama
```bash
# Download from https://ollama.ai or use:
curl -fsSL https://ollama.com/install.sh | sh
```

### 2. Pull Required Models
```bash
ollama pull nomic-embed-text
ollama pull llama3.2:1b

# For better accuracy (optional):
ollama pull llama3.2:3b
ollama pull llama3:8b
```

### 3. Verify Models
```bash
ollama ls
```

### 4. Install Python Dependencies
```bash
pip install langchain langchain-community langchain-ollama langchain-chroma pypdf python-dotenv requests
```

### 5. Prepare Data Directory
```bash
# Create data directory if it doesn't exist
mkdir -p ../data/

# Add your medical PDF files to ../data/
# Example: cp your-medical-pdfs/*.pdf ../data/
```

In [93]:
import warnings
warnings.filterwarnings('ignore')
import os
from dotenv import load_dotenv
load_dotenv()

True

## Step 1: Environment Setup
Import required libraries and load environment variables.

In [94]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain.prompts import PromptTemplate

## Configuration Management
Define all configuration parameters in one place for easy adjustment.

In [95]:
from dataclasses import dataclass
from typing import Optional
import logging

@dataclass
class ChatbotConfig:
    """Centralized configuration for medical chatbot"""
    
    # Model Configuration
    EMBEDDING_MODEL: str = "nomic-embed-text"
    LLM_MODEL: str = "llama3.2:1b"
    TEMPERATURE: float = 0.3
    
    # Chunking Configuration
    CHUNK_SIZE: int = 1000
    CHUNK_OVERLAP: int = 400
    
    # Retrieval Configuration
    RETRIEVAL_K: int = 5  # Retrieve top 3 documents
    SEARCH_TYPE: str = "similarity"  # or "mmr" for diversity
    
    # Database Configuration
    PERSIST_DIRECTORY: str = "db"
    DATA_DIRECTORY: str = "../data/"
    
    # Performance Configuration
    MAX_QUERY_LENGTH: int = 500
    REQUEST_TIMEOUT: int = 30
    
    # Safety Configuration
    ENABLE_MEDICAL_DISCLAIMER: bool = True
    ENABLE_QUERY_VALIDATION: bool = True
    ENABLE_EMERGENCY_DETECTION: bool = True
    
    # Logging Configuration
    LOG_LEVEL: str = "INFO"

# Initialize configuration
config = ChatbotConfig()

print("Configuration loaded successfully!")
print(f"LLM Model: {config.LLM_MODEL}")
print(f"Embedding Model: {config.EMBEDDING_MODEL}")
print(f"Chunk Size: {config.CHUNK_SIZE}")
print(f"Retrieval K: {config.RETRIEVAL_K}")

Configuration loaded successfully!
LLM Model: llama3.2:1b
Embedding Model: nomic-embed-text
Chunk Size: 1000
Retrieval K: 5


## Logging Setup
Configure structured logging for debugging and monitoring.

In [96]:
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=getattr(logging, config.LOG_LEVEL),
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f'chatbot_{datetime.now().strftime("%Y%m%d")}.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('MedicalChatbot')

# Test logging
logger.info("Medical Chatbot logging initialized")
logger.info(f"Log level: {config.LOG_LEVEL}")

print("Logging configured successfully!")

2026-02-15 18:53:21,285 - MedicalChatbot - INFO - Medical Chatbot logging initialized
2026-02-15 18:53:21,286 - MedicalChatbot - INFO - Log level: INFO


Logging configured successfully!


## Step 2: Load PDF Documents
Extract text from all PDF files in the data directory.

## Safety Utilities
Medical disclaimer, query validation, and emergency detection.

In [97]:
import re
from typing import Tuple, Optional

class SafetyValidator:
    """Validate queries and responses for medical safety"""
    
    # Emergency keywords that require immediate medical attention
    EMERGENCY_KEYWORDS = [
        'chest pain', 'heart attack', 'stroke', 'seizure', 'unconscious',
        'severe bleeding', 'difficulty breathing', 'can\'t breathe',
        'suicide', 'overdose', 'severe burn', 'choking', 'anaphylaxis'
    ]
    
    # Harmful intent keywords to block
    HARMFUL_KEYWORDS = [
        'how to die', 'kill myself', 'end my life', 'suicide methods',
        'self harm', 'cutting myself', 'overdose on'
    ]
    
    MEDICAL_DISCLAIMER = (
        "\n\nMEDICAL DISCLAIMER:\n"
        "This information is for educational purposes only and is not a substitute "
        "for professional medical advice, diagnosis, or treatment. Always seek the "
        "advice of your physician or qualified health provider with any questions "
        "regarding a medical condition. Never disregard professional medical advice "
        "or delay seeking it because of information provided here."
    )
    
    EMERGENCY_MESSAGE = (
        "\n\nEMERGENCY ALERT:\n"
        "Your question suggests a medical emergency. Please:\n"
        "• Call emergency services (911 in US, 999 in UK, 112 in EU) immediately\n"
        "• Go to the nearest emergency room\n"
        "• Contact your local emergency services\n\n"
        "Do NOT rely on this chatbot for emergency medical situations!"
    )
    
    @staticmethod
    def validate_query(query: str) -> Tuple[bool, Optional[str]]:
        """
        Validate if query is safe and appropriate
        Returns: (is_valid, error_message)
        """
        query_lower = query.lower().strip()
        
        # Check length
        if len(query) < 3:
            return False, "Query too short. Please ask a complete question."
        
        if len(query) > config.MAX_QUERY_LENGTH:
            return False, f"Query too long (max {config.MAX_QUERY_LENGTH} characters)."
        
        # Check for harmful intent
        for keyword in SafetyValidator.HARMFUL_KEYWORDS:
            if keyword in query_lower:
                logger.warning(f"Harmful query detected: {keyword}")
                return False, (
                    "I cannot provide information that could cause harm.\n\n"
                    "If you're experiencing a mental health crisis:\n"
                    "• National Suicide Prevention Lifeline: 988 (US)\n"
                    "• Crisis Text Line: Text HOME to 741741\n"
                    "• International: https://findahelpline.com"
                )
        
        return True, None
    
    @staticmethod
    def detect_emergency(query: str) -> bool:
        """Detect if query describes a medical emergency"""
        query_lower = query.lower()
        for keyword in SafetyValidator.EMERGENCY_KEYWORDS:
            if keyword in query_lower:
                logger.warning(f"Emergency keyword detected: {keyword}")
                return True
        return False
    
    @staticmethod
    def add_disclaimer(response: str, is_emergency: bool = False) -> str:
        """Add medical disclaimer to response"""
        if is_emergency:
            response = SafetyValidator.EMERGENCY_MESSAGE + "\n\n" + response
        
        if config.ENABLE_MEDICAL_DISCLAIMER:
            response += SafetyValidator.MEDICAL_DISCLAIMER
        
        return response

# Initialize validator
safety = SafetyValidator()

# Test validation
test_queries = [
    "What is diabetes?",
    "I have severe chest pain",
    "x",  # Too short
]

print("🛡️ Safety Validator initialized!\n")
print("Testing validation:")
for query in test_queries:
    is_valid, error = safety.validate_query(query)
    is_emergency = safety.detect_emergency(query)
    print(f"  Query: '{query[:30]}...'")
    print(f"  Valid: {is_valid}, Emergency: {is_emergency}")
    if error:
        print(f"  Error: {error[:50]}...")
    print()

2026-02-15 18:53:21,317 - MedicalChatbot - WARNING - Emergency keyword detected: chest pain


🛡️ Safety Validator initialized!

Testing validation:
  Query: 'What is diabetes?...'
  Valid: True, Emergency: False

  Query: 'I have severe chest pain...'
  Valid: True, Emergency: True

  Query: 'x...'
  Valid: False, Emergency: False
  Error: Query too short. Please ask a complete question....



## Response Quality & Metrics
Evaluate response quality, confidence scoring, and performance metrics.

In [98]:
from typing import Dict, List, Any
import json

class ResponseAnalyzer:
    """Analyze and score chatbot responses"""
    
    # Phrases indicating uncertainty
    UNCERTAINTY_PHRASES = [
        "i don't know",
        "not sure",
        "cannot find",
        "unclear",
        "insufficient information",
        "i don't have enough information"
    ]
    
    @staticmethod
    def calculate_confidence(result: Dict[str, Any]) -> float:
        """
        Calculate confidence score based on retrieval and response
        Returns: confidence score between 0 and 1
        """
        confidence = 0.5  # Base confidence
        
        # Check if we have source documents
        if result.get("source_documents"):
            docs = result["source_documents"]
            
            # More documents = higher confidence (up to 0.3 boost)
            doc_boost = min(len(docs) * 0.1, 0.3)
            confidence += doc_boost
            
            # Check document relevance scores if available
            # ChromaDB provides distance/similarity scores
            # Lower distance = higher relevance
        
        # Check response for uncertainty
        answer = result.get("result", "").lower()
        for phrase in ResponseAnalyzer.UNCERTAINTY_PHRASES:
            if phrase in answer:
                confidence -= 0.3
                break
        
        # Check response length (very short = low confidence)
        if len(answer) < 50:
            confidence -= 0.2
        
        # Ensure confidence is between 0 and 1
        confidence = max(0.0, min(1.0, confidence))
        
        return round(confidence, 2)
    
    @staticmethod
    def get_confidence_label(confidence: float) -> str:
        """Get human-readable confidence label"""
        if confidence >= 0.8:
            return "🟢 High Confidence"
        elif confidence >= 0.5:
            return "🟡 Medium Confidence"
        else:
            return "🔴 Low Confidence"
    
    @staticmethod
    def format_sources(source_documents: List[Any]) -> List[Dict[str, str]]:
        """Format source documents for display"""
        formatted_sources = []
        seen_sources = set()
        
        for doc in source_documents:
            source = doc.metadata.get("source", "Unknown")
            filename = source.split("/")[-1] if "/" in source else source
            
            # Avoid duplicate sources
            if filename not in seen_sources:
                formatted_sources.append({
                    "filename": filename,
                    "content_preview": doc.page_content[:150] + "..."
                })
                seen_sources.add(filename)
        
        return formatted_sources

class PerformanceMetrics:
    """Track performance metrics"""
    
    def __init__(self):
        self.metrics = {
            "total_queries": 0,
            "successful_queries": 0,
            "failed_queries": 0,
            "total_response_time": 0.0,
            "avg_response_time": 0.0,
            "avg_confidence": 0.0,
            "emergency_detections": 0
        }
    
    def record_query(self, success: bool, response_time: float, 
                    confidence: float = 0.0, is_emergency: bool = False):
        """Record query metrics"""
        self.metrics["total_queries"] += 1
        
        if success:
            self.metrics["successful_queries"] += 1
            self.metrics["total_response_time"] += response_time
            
            # Update averages
            queries = self.metrics["successful_queries"]
            self.metrics["avg_response_time"] = (
                self.metrics["total_response_time"] / queries
            )
            
            # Update average confidence
            prev_avg = self.metrics["avg_confidence"]
            self.metrics["avg_confidence"] = (
                (prev_avg * (queries - 1) + confidence) / queries
            )
        else:
            self.metrics["failed_queries"] += 1
        
        if is_emergency:
            self.metrics["emergency_detections"] += 1
    
    def get_summary(self) -> str:
        """Get formatted metrics summary"""
        m = self.metrics
        success_rate = (m["successful_queries"] / m["total_queries"] * 100 
                       if m["total_queries"] > 0 else 0)
        
        summary = f"""
        Performance Metrics:
        • Total Queries: {m['total_queries']}
        • Success Rate: {success_rate:.1f}%
        • Avg Response Time: {m['avg_response_time']:.2f}s
        • Avg Confidence: {m['avg_confidence']:.2%}
        • Emergencies Detected: {m['emergency_detections']}
        • Failed Queries: {m['failed_queries']}
        """
        return summary.strip()

# Initialize analyzers
analyzer = ResponseAnalyzer()
metrics = PerformanceMetrics()

print("  - Response analysis and metrics initialized!")
print("  - Ready to track:")
print("  - Confidence scoring")
print("  - Response quality")
print("  - Performance metrics")
print("  - Source formatting")

  - Response analysis and metrics initialized!
  - Ready to track:
  - Confidence scoring
  - Response quality
  - Performance metrics
  - Source formatting


In [99]:
# Extract text from PDF files in the 'data' directory
def load_pdf(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

extracted_data = load_pdf("../data/")

## Step 3: Filter Document Metadata
Keep only essential metadata (source) to reduce memory usage.

In [100]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [101]:
minimal_documents = filter_to_minimal_docs(extracted_data)

## Step 4: Split Documents into Chunks
Break down documents into smaller chunks for better retrieval and processing.

In [102]:
# Split the documents into smaller chunks
def text_split(minimal_documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE, 
        chunk_overlap=config.CHUNK_OVERLAP
    )
    text_chunks = text_splitter.split_documents(minimal_documents)
    return text_chunks

text_chunks = text_split(minimal_documents)
logger.info(f"Created {len(text_chunks)} text chunks")
print(f"Number of text chunks: {len(text_chunks)}")

2026-02-15 18:53:42,326 - MedicalChatbot - INFO - Created 5153 text chunks


Number of text chunks: 5153


## Step 5: Initialize Ollama Embeddings
Use the local `nomic-embed-text` model for document embeddings.

In [103]:
# Initialize Ollama embeddings with nomic-embed-text model
def download_ollama_embeddings():
    embeddings = OllamaEmbeddings(model=config.EMBEDDING_MODEL)
    return embeddings

embeddings = download_ollama_embeddings()
logger.info(f"Embeddings initialized with model: {config.EMBEDDING_MODEL}")
print(f"Ollama embeddings initialized successfully!")
print(f"   Model: {config.EMBEDDING_MODEL}")

2026-02-15 18:53:42,388 - MedicalChatbot - INFO - Embeddings initialized with model: nomic-embed-text


Ollama embeddings initialized successfully!
   Model: nomic-embed-text


## Step 6: Create Vector Store
Store document embeddings in ChromaDB for efficient similarity search.

## Step 7-10: Conversational RAG Setup (Single Unified Approach)

The next cells set up a **conversational RAG system with memory**:
- **Step 7:** Import conversational components (ConversationalRetrievalChain + ConversationBufferMemory)
- **Step 8:** Create memory object (stores chat history)
- **Step 9:** Create conversational prompt (includes {chat_history}, {context}, {question})
- **Step 10:** Initialize LLM + Create conversational chain

**Why single approach?** 
- ✅ No need for separate "basic" and "conversational" chains
- ✅ Conversational chain handles both simple questions AND follow-ups
- ✅ Simpler, cleaner, production-ready code
- ✅ Understands "it", "this", "that" by maintaining conversation context

In [104]:
# Initialize ChromaDB
persist_directory = config.PERSIST_DIRECTORY

try:
    vectordb = Chroma.from_documents(
        documents=text_chunks,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    logger.info(f"Vector store created with {len(text_chunks)} documents")
    print(f"Vector store created and persisted to '{persist_directory}'!")
    print(f"   Total documents indexed: {len(text_chunks)}")
except Exception as e:
    logger.error(f"Failed to create vector store: {e}")
    raise

2026-02-15 19:27:14,624 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2026-02-15 19:27:27,744 - MedicalChatbot - INFO - Vector store created with 5153 documents


Vector store created and persisted to 'db'!
   Total documents indexed: 5153


In [105]:
# Step 7: Import conversational components (with memory)
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory

print("✅ Conversational imports loaded!")
print("   - ConversationalRetrievalChain: Handles memory-aware retrieval")
print("   - ConversationBufferWindowMemory: Stores LAST 4 Q&A pairs (prevents overflow)")

✅ Conversational imports loaded!
   - ConversationalRetrievalChain: Handles memory-aware retrieval
   - ConversationBufferMemory: Stores full chat history


In [106]:
# Step 8: Create Conversation Memory (WITH WINDOW LIMIT - CRITICAL FIX!)
# Keeps only last 4 Q&A pairs (8 messages) to prevent context overflow
memory = ConversationBufferWindowMemory(
    k=4,  # 🔑 CRITICAL: Limit to last 4 Q&A pairs (8 messages)
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

logger.info("Conversation window memory initialized with k=4")
print("✅ Conversation Memory Created!")
print(f"   Type: ConversationBufferWindowMemory")
print(f"   Window Size: k=4 (last 4 Q&A pairs = 8 messages)")

print(f"   Purpose: Prevents memory overflow while maintaining context")print(f"   This fixes the 12-message problem!")

2026-02-15 19:27:27,759 - MedicalChatbot - INFO - Conversation memory initialized


✅ Conversation Memory Created!
   Type: ConversationBufferMemory
   Tracks: All questions + answers
   Purpose: Maintains context for follow-up questions


In [107]:
# Step 9: Create Conversational Prompt (with chat history)
conversational_prompt_template = """You are a cautious and reliable medical question-answering assistant with conversation memory.

STRICT RULES:
1. Answer ONLY using the provided context below
2. Use the chat history to understand follow-up questions (e.g., "it", "this", "that" refer to previous topics)
3. If the answer is not in the context, respond: 'I don't have enough information from the provided context.'
4. For emergencies, advise immediate medical care
5. Be clear, concise, and use simple language

Chat History:
{chat_history}

Context from documents:
{context}

Current Question:
{question}

Answer:"""

conversational_prompt = PromptTemplate(
    template=conversational_prompt_template,
    input_variables=["context", "question", "chat_history"]
)

logger.info("Conversational prompt template created")
print("✅ Conversational Prompt Created!")
print("   Includes: Chat history + Document context + Current question")
print("   Enables: Understanding of pronouns and references from previous messages")

2026-02-15 19:27:27,773 - MedicalChatbot - INFO - Conversational prompt template created


✅ Conversational Prompt Created!
   Includes: Chat history + Document context + Current question
   Enables: Understanding of pronouns and references from previous messages


In [108]:
# Step 10: Create Conversational Retrieval Chain
# This chain automatically:
# 1. Reformulates follow-up questions using chat history
# 2. Retrieves relevant documents
# 3. Generates answer with full context

# Initialize Ollama LLM
llm = ChatOllama(
    model=config.LLM_MODEL,
    temperature=config.TEMPERATURE,
)

try:
    conversational_qa = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectordb.as_retriever(
            search_type=config.SEARCH_TYPE,
            search_kwargs={'k': config.RETRIEVAL_K}
        ),
        memory=memory,
        return_source_documents=True,
        verbose=True,  # Enable to debug query reformulation
        combine_docs_chain_kwargs={"prompt": conversational_prompt}
    )
    
    logger.info("Conversational RAG chain initialized successfully")
    print("🎉 Conversational RAG Chain Created Successfully!")
    print(f"   LLM Model: {config.LLM_MODEL}")
    print(f"   Temperature: {config.TEMPERATURE}")
    print(f"   Retrieval K: {config.RETRIEVAL_K}")
    print(f"   Memory: Enabled ✅")
    print("\n📝 KEY IMPROVEMENTS:")
    print("   ✅ Remembers conversation history")
    print("   ✅ Reformulates follow-up questions with context")  
    print("   ✅ Maintains topic continuity")
    print("   ✅ Retrieves correct documents for 'it', 'this', 'that'")
    print("\n💡 This is the ONLY chain you need - no separate 'qa' chain!")
    
except Exception as e:
    logger.error(f"Failed to create conversational chain: {e}")
    print(f"❌ Error: {e}")
    raise

2026-02-15 19:27:27,832 - MedicalChatbot - INFO - Conversational RAG chain initialized successfully


🎉 Conversational RAG Chain Created Successfully!
   LLM Model: llama3.2:1b
   Temperature: 0.3
   Retrieval K: 5
   Memory: Enabled ✅

📝 KEY IMPROVEMENTS:
   ✅ Remembers conversation history
   ✅ Reformulates follow-up questions with context
   ✅ Maintains topic continuity
   ✅ Retrieves correct documents for 'it', 'this', 'that'

💡 This is the ONLY chain you need - no separate 'qa' chain!


In [ ]:
import time
import uuid
from datetime import datetime

def conversational_chat():
    """
    Conversational chatbot with memory - FIXES follow-up question problem!
    
    DEPENDENCIES: config, logger, conversational_qa
    Features:
    - Remembers conversation history
    - Reformulates follow-up questions
    - Maintains topic continuity
    - Full logging and error handling
    """
    # Verify dependencies
    try:
        _ = (config, logger, conversational_qa)
    except NameError as e:
        print(f"❌ ERROR: Missing dependency - {e}")
        print("Please ensure you've run the required cells first!")
        return
    
    session_id = str(uuid.uuid4())[:8]
    conversation_count = 0
    
    logger.info(f"Starting conversational chat session: {session_id}")
    
    print("=" * 80)
    print("🏥 Medical Chatbot - Conversational Mode (WITH MEMORY)")
    print("=" * 80)
    print(f"Session ID: {session_id}")
    print(f"Model: {config.LLM_MODEL}")
    print("\nCommands:")
    print("   • 'exit/quit/bye' - End conversation")
    print("   • 'history' - Show conversation summary")
    print("   • 'memory' - View current memory state")
    print("   • 'reset' - Clear conversation memory (fresh start)")
    print("   • 'clear' - Clear screen")
    print("   • 'export' - Export conversation to file")
    print("=" * 80)
    
    while True:
        try:
            user_query = input("\n🧑 You: ").strip()
            
            # Handle commands
            if user_query.lower() in ['exit', 'quit', 'bye', 'q']:
                print(f"\n👋 Conversation ended. Total queries: {conversation_count}")
                logger.info(f"Conversational session {session_id} ended. Total queries: {conversation_count}")
                break
            
            if user_query.lower() == 'history':
                # Show memory summary
                memory_vars = conversational_qa.memory.load_memory_variables({})
                chat_history = memory_vars.get('chat_history', [])
                
                if chat_history:
                    print("\n📜 Conversation Summary:")
                    print("=" * 60)
                    for i in range(0, len(chat_history), 2):
                        q_num = i // 2 + 1
                        if i < len(chat_history):
                            print(f"\n[{q_num}] Q: {chat_history[i].content[:80]}...")
                        if i + 1 < len(chat_history):
                            print(f"    A: {chat_history[i+1].content[:80]}...")
                    print("=" * 60)
                else:
                    print("\n📜 No conversation history yet.")
                continue
            
            if user_query.lower() == 'memory':
                # Show raw memory
                memory_vars = conversational_qa.memory.load_memory_variables({})
                chat_history = memory_vars.get('chat_history', [])
                print(f"\n🧠 Memory State:")
                print(f"   Messages in memory: {len(chat_history)}")
                print(f"   Queries processed: {conversation_count}")
                continue
            
            if user_query.lower() == 'clear':
                print("\n" * 50)
                continue
            
            if user_query.lower() == 'reset':
                # Clear conversation memory
                conversational_qa.memory.clear()
                conversation_count = 0
                print("\n🔄 Memory cleared! Starting fresh conversation.")
                logger.info(f"Memory reset in session {session_id}")
                continue
            
            if user_query.lower() == 'export':
                if conversation_count > 0:
                    filename = f"chat_conversational_{session_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
                    try:
                        memory_vars = conversational_qa.memory.load_memory_variables({})
                        chat_history = memory_vars.get('chat_history', [])
                        
                        with open(filename, 'w', encoding='utf-8') as f:
                            f.write(f"Medical Chatbot - Conversational Chat History\n")
                            f.write(f"Session ID: {session_id}\n")
                            f.write(f"Exported: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                            f.write("=" * 80 + "\n\n")
                            
                            for i in range(0, len(chat_history), 2):
                                q_num = i // 2 + 1
                                if i < len(chat_history):
                                    f.write(f"\n[Query {q_num}]\n")
                                    f.write(f"Q: {chat_history[i].content}\n\n")
                                if i + 1 < len(chat_history):
                                    f.write(f"A: {chat_history[i+1].content}\n\n")
                                f.write("-" * 80 + "\n")
                        
                        print(f"✅ Conversation exported to: {filename}")
                        logger.info(f"Chat history exported to {filename}")
                    except Exception as e:
                        print(f"❌ Error exporting: {e}")
                        logger.error(f"Failed to export chat history: {e}")
                else:
                    print("📜 No conversation to export yet.")
                continue
            
            if not user_query:
                continue
            
            # Log and display user input clearly
            print(f"\n{'='*80}")
            print(f"🧑 You: {user_query}")
            print(f"{'='*80}")
            logger.info(f"USER INPUT #{conversation_count + 1}: {user_query}")
            
            # Process query with conversation memory
            start_time = time.time()
            
            logger.info(f"Processing query #{conversation_count + 1} in session {session_id}: {user_query[:100]}")
            
            try:
                # Use conversational_qa which automatically handles memory
                result = conversational_qa({"question": user_query})
                answer = result["answer"]
                
                end_time = time.time()
                response_time = end_time - start_time
                
                # Log query processing details
                logger.info(f"Response generated in {response_time:.2f}s")
                logger.info(f"Retrieved {len(result.get('source_documents', []))} documents")
                
                # Display response with clear formatting
                print(f"\n{'='*80}")
                print(f"🤖 Assistant:")
                print(f"{'='*80}")
                print(answer)
                print(f"{'='*80}")
                
                conversation_count += 1
                
                # Show sources
                if result.get("source_documents"):
                    sources = [doc.metadata.get("source", "Unknown").split("/")[-1] 
                              for doc in result["source_documents"]]
                    unique_sources = list(dict.fromkeys(sources))  # Remove duplicates
                    print(f"Sources: {', '.join(unique_sources)}")
                
                print(f"Response Time: {response_time:.2f}s | Query #{conversation_count}")
                logger.info(f"OUTPUT ANSWER: {answer[:100]}...")
                print()
                
                logger.info(f"Query #{conversation_count} processed successfully in {response_time:.2f}s")
                
            except Exception as e:
                error_msg = str(e)
                print(f"\n❌ Error: {error_msg}")
                logger.error(f"Query processing error in session {session_id}: {error_msg}", exc_info=True)
                print("Please try rephrasing your question or type 'exit' to quit.")
            
        except KeyboardInterrupt:
            print(f"\n\nSession {session_id} interrupted. Goodbye!")
            logger.info(f"Conversational session {session_id} interrupted")
            break
        except Exception as e:
            logger.error(f"Unexpected error in session {session_id}: {e}", exc_info=True)
            print(f"\n❌ Unexpected error: {e}")
            print("Please try again or type 'exit' to quit.")
    
    # Final summary
    if conversation_count > 0:
        print(f"\nSession Summary:")
        print(f"   Total queries: {conversation_count}")
        print(f"   Session ID: {session_id}")
        
        # Check memory state
        memory_vars = conversational_qa.memory.load_memory_variables({})
        chat_history = memory_vars.get('chat_history', [])
        print(f"   Messages in memory: {len(chat_history)}")

## Test Chatbot

In [119]:
try:
    conversational_qa.memory.clear()
    print("✅ Memory cleared successfully!")
    print("📊 All conversation history removed.")
    print("🔄 You can start a fresh conversation now.")
    
    # Verify it's empty
    memory_vars = conversational_qa.memory.load_memory_variables({})
    chat_history = memory_vars.get('chat_history', [])
    print(f"Current memory messages: {len(chat_history)} (should be 0)")
    
except Exception as e:
    print(f"Error clearing memory: {e}")
    print("Make sure you've run Cell 29 (chain creation) first!")

✅ Memory cleared successfully!
📊 All conversation history removed.
🔄 You can start a fresh conversation now.
🧠 Current memory messages: 0 (should be 0)


In [120]:
conversational_chat()

2026-02-15 20:02:56,226 - MedicalChatbot - INFO - Starting conversational chat session: 3ae870e7


🏥 Medical Chatbot - Conversational Mode (WITH MEMORY)
Session ID: 3ae870e7
Model: llama3.2:1b

Commands:
   • 'exit/quit/bye' - End conversation
   • 'history' - Show conversation summary
   • 'memory' - View current memory state
   • 'reset' - Clear conversation memory (fresh start)
   • 'clear' - Clear screen
   • 'export' - Export conversation to file
📜 No conversation to export yet.
📜 No conversation to export yet.


2026-02-15 20:03:31,263 - MedicalChatbot - INFO - USER INPUT #1: What is corneal transplantation?
2026-02-15 20:03:31,264 - MedicalChatbot - INFO - Processing query #1 in session 3ae870e7: What is corneal transplantation?



🧑 You: What is corneal transplantation?


2026-02-15 20:03:31,864 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2026-02-15 20:03:40,810 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-15 20:03:43,539 - MedicalChatbot - INFO - Response generated in 12.28s
2026-02-15 20:03:43,540 - MedicalChatbot - INFO - Retrieved 5 documents
2026-02-15 20:03:43,540 - MedicalChatbot - INFO - OUTPUT ANSWER: Corneal transplantation is a surgical procedure where a damaged or diseased cornea (the transparent ...
2026-02-15 20:03:43,541 - MedicalChatbot - INFO - Query #1 processed successfully in 12.28s


...


🧑 You: What are the different types of this procedure?


2026-02-15 20:04:27,649 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-15 20:04:29,014 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2026-02-15 20:04:39,853 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-15 20:04:42,914 - MedicalChatbot - INFO - Response generated in 16.57s
2026-02-15 20:04:42,915 - MedicalChatbot - INFO - Retrieved 5 documents
2026-02-15 20:04:42,916 - MedicalChatbot - INFO - OUTPUT ANSWER: The main difference between corneal transplantation and other treatments for severe eye damage or di...
2026-02-15 20:04:42,916 - MedicalChatbot - INFO - Query #2 processed successfully in 16.57s



🤖 Assistant:
The main difference between corneal transplantation and other treatments for severe eye damage or disease is the type of donor tissue used. Corneal transplantation uses a healthy donor cornea from another person's eye, while other treatments may use artificial lenses (corrective lenses) or other types of surgery to repair damaged eyes.
📚 Sources: books.pdf
⏱️  Response Time: 16.57s | Query #2



2026-02-15 20:05:23,948 - MedicalChatbot - INFO - Conversational session 3ae870e7 ended. Total queries: 2



👋 Conversation ended. Total queries: 2

📊 Session Summary:
   Total queries: 2
   Session ID: 3ae870e7
   Messages in memory: 4


## Optional: Clear Old Database
If you need to recreate the vector database with new embeddings, run this cell to delete the old database.

In [111]:
import shutil
import os

def clear_database():
    """
    Clear the existing ChromaDB database to recreate with new embeddings.
    """
    db_path = "db"
    try:
        if os.path.exists(db_path):
            shutil.rmtree(db_path)
            print(f"Database '{db_path}' has been cleared successfully!")
        else:
            print(f"ℹDatabase '{db_path}' does not exist. No need to clear.")
    except Exception as e:
        print(f"Error clearing database: {e}")

# Uncomment the line below to clear the database
# clear_database()